# Autenticação de Entrada (Inbound Auth)

AgentCore Identity permite validar acesso de entrada (Inbound Auth) para usuários e aplicações chamando agentes ou ferramentas em um AgentCore Runtime ou validar acesso a alvos AgentCore Gateway. Ele também fornece acesso seguro de saída (Outbound Auth) de um agente para serviços externos ou um alvo Gateway. Ele se integra com seus provedores de identidade existentes (como Amazon Cognito) enquanto impõe limites de permissão para agentes atuando independentemente ou em nome de usuários (via OAuth).

Inbound Auth valida chamadores tentando invocar agentes ou ferramentas, seja hospedados em AgentCore Runtime, AgentCore Gateway ou em outros ambientes. Inbound Auth funciona com IAM (credenciais SigV4) ou com autorização OAuth.

Por padrão, Amazon Bedrock AgentCore usa credenciais IAM, significando que requisições de usuário ao agente são autenticadas com as credenciais IAM do usuário. Se você usar OAuth, precisará especificar o seguinte ao configurar seus recursos AgentCore Runtime ou endpoints AgentCore Gateway:

- URL do servidor de descoberta OAuth — Uma string que deve corresponder ao padrão ^.+/\.well-known/openid-configuration$ para URLs de descoberta OpenID Connect

- Audiências permitidas — Lista de audiências permitidas para tokens JWT

- Clientes permitidos — Lista de identificadores de cliente permitidos

Se você usar a CLI AgentCore, pode especificar o tipo de autorização (e servidor de descoberta OAuth) para um AgentCore Runtime quando usar o comando **configure**. Você também pode usar a operação CreateAgentRuntime e o console Amazon Bedrock AgentCore. Se estiver criando um Gateway, use a operação CreateGateway ou o console.

Antes do usuário poder usar o agente, a aplicação cliente deve fazer o usuário autenticar com o autorizador OAuth. Seu cliente recebe um bearer token que então passa para o agente em uma requisição de invocação. Ao receber, o agente valida o token com o servidor de autorização antes de permitir acesso.


## Visão Geral

Neste tutorial, modificaremos o agente que você implantou em 01-AgentCore-runtime e o configuraremos para Inbound Auth usando Cognito como provedor de identidade. Você configurará um User Pool Cognito com um usuário e um app client. Você aprenderá como hospedar seu agente existente, usando Amazon Bedrock AgentCore Runtime com Inbound Auth usando o user pool Cognito.

### Arquitetura do Tutorial

<div style="text-align:center">
    <img src="images/inbound_auth_cognito.png" width="90%"/>
</div>

### Detalhes do Tutorial


| Informação          | Detalhes                                                                            |
|:--------------------|:------------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                      |
| Tipo de agente      | Único                                                                               |
| Framework Agêntico  | Strands Agents                                                                      |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                          |
| Componentes         | Hospedagem de agente no AgentCore Runtime. Usando Strands Agent e Amazon Bedrock Model |
| Vertical            | Cross-vertical                                                                      |
| Complexidade        | Fácil                                                                               |
| Inbound Auth        | Cognito                                                                             |
| SDK usado           | Amazon BedrockAgentCore Python SDK e boto3                                          |



### Funcionalidades Chave do Tutorial

* Hospedagem de Agentes no Amazon Bedrock AgentCore Runtime com Inbound Auth usando Amazon Cognito
* Uso de modelos Amazon Bedrock
* Uso de Strands Agents

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker em execução

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Configurando Amazon Cognito para Autenticação

Vamos provisionar um Userpool Cognito com um App client e um usuário de teste. Usaremos Amazon Cognito para fornecer tokens JWT para acessar nosso servidor MCP implantado. Para fazer isso, usaremos a função de suporte `setup_cognito_user_pool` do nosso script `utils`.

Nota: O access_token do Cognito é válido apenas por 2 horas. Se o access_token expirar, você pode gerar outro access_token usando o método `reauthenticate_user`.

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(
    os.path.abspath("__file__" if "__file__" in globals() else ".")
)

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import setup_cognito_user_pool, reauthenticate_user

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")

## Preparando seu agente para implantação no AgentCore Runtime

### Strands Agents com modelo Amazon Bedrock
Vamos começar com nosso Strands Agent que criamos no tutorial 01-AgentCore-runtime e configurá-lo com Inbound Auth que usa Amazon Cognito como Provedor de Identidade.

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## Implantando o agente no AgentCore Runtime

A operação `CreateAgentRuntime` suporta opções abrangentes de configuração, permitindo que você especifique imagens de container, variáveis de ambiente e configurações de criptografia. Você também pode configurar definições de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente.

**Nota:** A melhor prática de operações é empacotar código como container e enviar para ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCore Python SDK para facilmente empacotar seus artefatos e implantá-los no AgentCore runtime.

### Configurar implantação AgentCore Runtime

Em seguida, usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR na inicialização.

Durante a etapa de configuração, seu docker file será gerado com base no código da sua aplicação

**Importante** - Atualize a URL de descoberta Cognito e o ID do App client Cognito das etapas anteriores.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
region

discovery_url = cognito_config.get("discovery_url")

client_id = cognito_config.get("client_id")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_inbound_identity",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
)
response

## Revisar a configuração do AgentCore

In [ ]:
!cat .bedrock_agentcore.yaml

### Iniciando agente no AgentCore Runtime

Agora que temos um docker file, vamos iniciar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

### Verificando o Status do AgentCore Runtime
Agora que implantamos o AgentCore Runtime, vamos verificar seu status de implantação

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### Invocando AgentCore Runtime sem autorização

Finalmente, podemos invocar nosso AgentCore Runtime com um payload. Tente executar a célula a seguir e você verá um erro que diz **"AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type".**

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"})
invoke_response

### Invocando AgentCore Runtime com autorização

Vamos invocar o agente com o tipo de token de autorização correto. No nosso caso, será o access token do Cognito. Copie o access token da célula "**Provision a Cognito User Pool**"

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"))
invoke_response = agentcore_runtime.invoke(
    {"prompt": "How is the weather now?"}, bearer_token=bearer_token
)
invoke_response

## Limpeza (Opcional)

Vamos agora limpar o AgentCore Runtime criado

In [ ]:
from boto3.session import Session
import boto3

boto_session = Session()

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split("/")[1], force=True
)

# Parabéns!